# 4.19 t-SNE

t-SNE turns unlabeled data into structure by choosing the right notion of similarity, compression, or surprise.

Part 4 moves from prediction with labels to discovery without labels. Probability normalization and distance-based kernels become the way local influence is measured. t-SNE builds a 2D map by matching neighbor probabilities, so it is useful for exploration but not for declaring ground-truth clusters.

Save a copy to Drive to edit.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_digits
from sklearn.decomposition import NMF
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap
from sklearn.manifold import LocallyLinearEmbedding
from sklearn.manifold import MDS
from sklearn.manifold import SpectralEmbedding
from sklearn.manifold import TSNE
from sklearn.manifold import trustworthiness
from sklearn.metrics import pairwise_distances
from sklearn.random_projection import GaussianRandomProjection
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

def dimred_ladder():
    """D1..D5 dimensionality-reduction ladder. Returns [(name, X, y), ...] of rising ambient dim.

    2-D toy -> 3-D swiss-roll-ish -> digits(64-D) -> the same with noise dims -> a wide feature set.
    y is a color/label for visualization only.
    """
    rungs = []
    rng = np.random.default_rng(3)

    t = np.linspace(0, 4, 120)
    x1 = np.column_stack([t, 0.5 * t + rng.normal(0, 0.05, 120)])
    rungs.append(("D1 near-1-D line in 2-D", x1, t))

    tt = np.linspace(0, 3 * np.pi, 200)
    x2 = np.column_stack([tt * np.cos(tt), 8 * rng.random(200), tt * np.sin(tt)])
    rungs.append(("D2 swiss-roll (3-D)", x2, tt))

    digits = load_digits()
    rungs.append(("D3 digits (real, 64-D)", digits.data / 16.0, digits.target))

    xn = np.hstack([digits.data / 16.0, rng.normal(0, 1, size=(digits.data.shape[0], 32))])
    rungs.append(("D4 digits + 32 noise dims", xn, digits.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (30-D)", bc.data, bc.target))

    return rungs



def sample_for_embedding(X, y, max_points=500, seed=SEED):
    rng = np.random.default_rng(seed)
    if X.shape[0] <= max_points:
        return X, y
    idx = rng.choice(X.shape[0], size=max_points, replace=False)
    order = np.sort(idx)
    return X[order], y[order]


def standardize_for_geometry(X):
    scaler = StandardScaler()
    return scaler.fit_transform(X)


def nonnegative_for_factorization(X):
    scaler = MinMaxScaler()
    return scaler.fit_transform(X)


def safe_trustworthiness(X, Z):
    neighbors = min(10, max(1, (X.shape[0] - 1) // 3))
    return float(trustworthiness(X, Z, n_neighbors=neighbors))


def plot_ladder_embeddings(results, metric_name):
    fig, axes = plt.subplots(1, len(results), figsize=(17, 3.4))
    for ax, item in zip(axes, results):
        scatter = ax.scatter(item["Z"][:, 0], item["Z"][:, 1], c=item["y"], s=10, cmap="viridis")
        ax.set_title(item["name"].split("(")[0], fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle("2D embeddings across the D1-D5 ladder")
    plt.show()

    fig, ax = plt.subplots(figsize=(6, 3.5))
    xs = np.arange(1, len(results) + 1)
    ys = [item["metric"] for item in results]
    ax.plot(xs, ys, marker="o")
    ax.set_xticks(xs)
    ax.set_xticklabels([f"D{i}" for i in xs])
    ax.set_ylabel(metric_name)
    ax.set_xlabel("dataset rung")
    ax.set_title(f"{metric_name} vs. ladder complexity")
    ax.grid(True, alpha=0.3)
    plt.show()


def preview_ladder(rungs):
    for i, (name, X, y) in enumerate(rungs, start=1):
        unique = np.unique(y)
        label_info = len(unique) if unique.size < 30 else "continuous"
        print(f"D{i}: {name} | X={X.shape} | color/label info={label_info}")
        print(np.round(X[:3, :min(5, X.shape[1])], 3))

## The concept, built once: local probability from distances

The lesson's local kernel is $$p_i=rac{\exp(-d_i^2/2h^2)}{\sum_j\exp(-d_j^2/2h^2)}$$. With $h=1$ and distances $[1,0,1]$, the middle point receives about $0.452$ of the influence.

In [ ]:
def local_kernel_weights(distances, h=1.0):
    distances = np.asarray(distances, dtype=float)
    weights = np.exp(-(distances ** 2) / (2.0 * h ** 2))
    probabilities = weights / weights.sum()
    return weights, probabilities

weights, probabilities = local_kernel_weights([1.0, 0.0, 1.0], h=1.0)
print("weights", np.round(weights, 3))
print("sum", round(float(weights.sum()), 3))
print("middle influence", round(float(probabilities[1]), 3))
assert np.allclose(np.round(weights, 3), [0.607, 1.0, 0.607])
assert np.isclose(round(float(weights.sum()), 3), 2.213)
assert np.isclose(round(float(probabilities[1]), 3), 0.452)

Now build the reusable method. We explicitly compute pairwise distances for the audit trail, then call sklearn's real `TSNE` implementation for the optimized embedding.

In [ ]:
def method(X, perplexity=30, seed=SEED):
    X_scaled = standardize_for_geometry(X)
    distances = pairwise_distances(X_scaled[:5])
    print("audit pairwise block shape", distances.shape)
    safe_perplexity = min(float(perplexity), max(2.0, (X_scaled.shape[0] - 1) / 3.0))
    model = TSNE(n_components=2, perplexity=safe_perplexity, init="pca", learning_rate="auto", max_iter=500, random_state=seed, method="barnes_hut")
    return model.fit_transform(X_scaled)

assert pairwise_distances(np.array([[0.0], [1.0], [2.0]])).shape == (3, 3)

## The dataset ladder

We use the shared F3 dimensionality-reduction ladder: a near-1D toy line, a 3D swiss-roll-style surface, real handwritten digits, digits with added noise dimensions, and a real 30-dimensional breast-cancer feature table. The labels are only colors for visualization; the embedding method does not train on them.

In [ ]:
rungs = dimred_ladder()
preview_ladder(rungs)

## Run the same method across D1-D5

Each rung is standardized or scaled in the same way, embedded into 2D, and scored with trustworthiness. Subsampling is seeded and only bounds future notebook runtime; this build script does not execute the notebook.

In [ ]:
metric_name = "trustworthiness"
results = []
for rung_index, (name, X, y) in enumerate(rungs, start=1):
    X_small, y_small = sample_for_embedding(X, y, max_points=450, seed=SEED + rung_index)
    Z = method(X_small, perplexity=25, seed=SEED + rung_index)
    score = safe_trustworthiness(standardize_for_geometry(X_small), Z)
    results.append({"name": name, "X": X_small, "y": y_small, "Z": Z, "metric": score})
    print(f"D{rung_index} | {name:32s} | trustworthiness={score:.3f}")

## Results visualization

The closing figure has two parts: small-multiple embedding panels for D1-D5, then a metric curve as the data become more realistic and higher dimensional.

In [ ]:
plot_ladder_embeddings(results, metric_name)

## Pitfall on the hardest rung: visual islands are not true clusters

t-SNE is optimized for local neighbor probabilities. On D5, changing seed or perplexity can move islands around; the fix is to report trustworthiness and neighbor checks instead of naming clusters by eye.

In [ ]:
name, X, y = rungs[-1]
X_small, y_small = sample_for_embedding(X, y, max_points=450, seed=19)
Z_low = method(X_small, perplexity=5, seed=0)
Z_high = method(X_small, perplexity=50, seed=1)
low_score = safe_trustworthiness(standardize_for_geometry(X_small), Z_low)
high_score = safe_trustworthiness(standardize_for_geometry(X_small), Z_high)
print("perplexity 5 trustworthiness", round(low_score, 3))
print("perplexity 50 trustworthiness", round(high_score, 3))
print("fix: compare scores and rerun neighbor audits before interpreting islands")

## Evaluate it + Practice

- Report trustworthiness next to a no-skill baseline such as plotting two raw standardized features or a random 2D map.
- Sanity check that nearby points in the original space still have nearby points in the embedding.
- Ablate the key idea: remove scaling, change the random seed, or use too few neighbors/components and watch the metric move.
- Watch for failure signals: unstable layouts, one feature dominating distances, disconnected neighbor graphs, or a better objective with worse held-out structure.
- Treat labels as a post-hoc audit only; unsupervised methods do not get to train on them.


### Practice

Try perplexities 5, 15, 30, and 50 on D3; record trustworthiness for each.

In [ ]:
# Your code here


### Practice

Color D5 by the provided diagnostic label only after computing the embedding; what might be over-interpreted?

In [ ]:
# Your code here


### Practice

Compare t-SNE with random projection on the same D4 sample.

In [ ]:
# Your code here
